# Workflow Complet : ResNet50V2 sur GTEx 11 Classes (Expérience A)

**Protocole d'Entraînement :**
- Backbone : `ResNet50V2` (poids ImageNet)
- Input : `224x224x3` (RGB, float32, `[0,255]`)
- Classes : 11 (Bladder, Brain, Cerebellum, Kidney, Liver, Lung, Muscle, Oesophagus, Pancreas, Spleen, Testis)
- Stratégie : 3 phases (Tête, 30% Fine-Tuning, Full Fine-Tuning)
- Précision Mixte : `mixed_float16`
- Environnement cible : Kaggle


## ⚠️ Avertissement Légal et Médical
Ce modèle est développé exclusivement à des fins de **recherche éducative et de démonstration technique**. 
Il ne s'agit pas d'un dispositif médical certifié. Les prédictions générées par ce réseau neuronal ne doivent en **aucun cas** être utilisées pour le diagnostic clinique, la prise de décision thérapeutique, ou remplacer le jugement d'un médecin anatomopathologiste qualifié.


In [ ]:
REPO_URL = "https://github.com/MyElhadri/histology-ai-classification.git"
BRANCH = "main"

PROJECT_DIR = "/kaggle/working/histology-ai-classification"
OUTPUT_DIR = "/kaggle/working/resnet50v2-gtex-11-exp-a"

CONFIG_PATH = (
    "configs/experiments/"
    "resnet50v2_gtex_11_exp_a.yaml"
)

RUN_TESTS = True
RUN_DATASET_AUDIT = True
RUN_DRY_RUN = True
RUN_SMOKE_TEST = False
RUN_TRAINING = True
RUN_VALIDATION_EVALUATION = True
RUN_FINAL_TEST = False
GENERATE_REPORT = True
RESUME_TRAINING = True
ALLOW_OVERWRITE = False


In [ ]:
import os
import sys

in_kaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''
print(f"Environnement Kaggle détecté : {in_kaggle}")


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU détecté : {gpus[0]}")
    !nvidia-smi
else:
    print("⚠️ AUCUN GPU DÉTECTÉ !")


In [ ]:
if gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Précision mixte (mixed_float16) activée avec succès.")
else:
    print("Mode CPU / T4 non optimal, mixed_float16 non activé.")


In [ ]:
print("Contenu de /kaggle/input :")
!ls -la /kaggle/input


In [ ]:
from pathlib import Path

DATASET_DIR = None
matches = list(Path("/kaggle/input").rglob("GTEx_11_classes"))

if len(matches) == 1:
    DATASET_DIR = matches[0]
    print(f"Dataset GTEx détecté automatiquement : {DATASET_DIR}")
elif len(matches) > 1:
    print("Plusieurs dossiers GTEx_11_classes détectés :")
    for m in matches:
        print(f" - {m}")
    raise RuntimeError("Erreur : Impossible de déterminer quel dataset utiliser.")
else:
    print("Dossiers actuels sous /kaggle/input :")
    for root, dirs, files in os.walk("/kaggle/input"):
        for d in dirs:
            print(os.path.join(root, d))
    raise RuntimeError("Erreur : GTEx_11_classes introuvable. Veuillez attacher le dataset au notebook.")


In [ ]:
if not os.path.exists(PROJECT_DIR):
    print(f"Clonage du dépôt GitHub depuis {REPO_URL} (branche {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print(f"Le dépôt existe déjà. Mise à jour (git pull)...")
    !cd {PROJECT_DIR} && git fetch && git reset --hard origin/{BRANCH}


In [ ]:
import subprocess

def get_git_commit():
    try:
        return subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=PROJECT_DIR, text=True).strip()
    except:
        return "Inconnu"

print(f"Commit actuel : {get_git_commit()}")


In [ ]:
print("Installation des dépendances manquantes (Kaggle préinstalle déjà TensorFlow)...")
!pip install -q pytest pyyaml scikit-learn pandas matplotlib


In [ ]:
if RUN_DATASET_AUDIT:
    print("Vérification de l'arborescence du dataset :")
    !ls -la {DATASET_DIR}
    !ls -la {DATASET_DIR}/metadata
else:
    print("Audit d'arborescence ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_DATASET_AUDIT:
    print("Lancement de l'audit d'intégrité GTEx (comptes, class_mapping, et isolation des donneurs)...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    audit_cmd = [
        sys.executable, "-u", "-c", 
        f"from src.data.gtex_integrity import audit_gtex_dataset; "
        f"audit_gtex_dataset('{DATASET_DIR}', '{OUTPUT_DIR}/dataset_integrity.json')"
    ]
    
    process = subprocess.Popen(
        audit_cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
        
    process.stdout.close()
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError(f"L'audit du dataset a échoué (Code {return_code}). Entraînement bloqué.")
    print("\nAudit du dataset réussi avec succès.")
else:
    print("RUN_DATASET_AUDIT = False. Ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_TESTS:
    print("Lancement de la suite de tests unitaires ciblés ResNet50V2...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["MPLBACKEND"] = "Agg"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    
    test_cmd = [
        sys.executable, "-u", "-m", "pytest",
        "tests/test_resnet50v2_gtex.py",
        "tests/test_gtex_dataset_integrity.py",
        "tests/test_resnet50v2_gtex_report.py",
        "-vv", "-s", "--tb=long", "--maxfail=1"
    ]
    
    process = subprocess.Popen(
        test_cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    failed_test_info = None
    
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
        if line.startswith("FAILED tests/"):
            failed_test_info = line.strip()
            
    process.stdout.close()
    return_code = process.wait()
    
    if return_code != 0:
        print("\n" + "="*70)
        print("LES TESTS ONT ÉCHOUÉ")
        print("="*70)
        if failed_test_info:
            parts = failed_test_info.split(" - ")[0].replace("FAILED ", "").split("::")
            if len(parts) >= 2:
                print(f"Fichier en échec : {parts[0]}")
                print(f"Test en échec : {parts[-1]}")
        print(f"Code de retour : {return_code}")
        print("INTERDICTION DE POURSUIVRE L'ENTRAÎNEMENT. Corrigez le code source d'abord.")
        print("="*70 + "\n")
        raise RuntimeError(f"Pytest a échoué avec le code {return_code}.")
        
    print("\nTous les tests unitaires ont réussi avec succès.")
else:
    print("RUN_TESTS = False. Étape de test ignorée.")


In [ ]:
import subprocess
import os
import sys

if RUN_DRY_RUN:
    print("Exécution du Dry-Run pour valider la configuration...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    
    cmd = [
        sys.executable, "-u", "scripts/train_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", OUTPUT_DIR,
        "--dry-run"
    ]
    
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
        
    process.stdout.close()
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("Le Dry-Run a échoué.")
    print("\nDry-Run validé.")
else:
    print("RUN_DRY_RUN = False. Ignoré.")


In [ ]:
if os.path.exists(OUTPUT_DIR):
    print("Le dossier résultats existe. Contenu :")
    !ls -la {OUTPUT_DIR}
else:
    print("Le dossier résultats n'existe pas encore.")


In [ ]:
import subprocess
import os
import sys

if RUN_TRAINING:
    print("Démarrage de l'entraînement (3 Phases)...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    
    cmd = [
        sys.executable, "-u", "scripts/train_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", OUTPUT_DIR
    ]
    
    if RUN_SMOKE_TEST:
        cmd.append("--smoke-test")
        print(">>> MODE SMOKE TEST ACTIVÉ <<<")
        
    if RESUME_TRAINING:
        cmd.append("--resume")
        
    if ALLOW_OVERWRITE:
        cmd.append("--overwrite")
        
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
        
    process.stdout.close()
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("L'entraînement a échoué.")
    print("\nEntraînement terminé avec succès.")
else:
    print("RUN_TRAINING = False. Ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_VALIDATION_EVALUATION:
    print("Démarrage de l'évaluation sur le split Validation...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    
    output_target = f"{OUTPUT_DIR}/smoke_test" if RUN_SMOKE_TEST else OUTPUT_DIR
    best_model = f"{output_target}/checkpoints/best_model.keras"
    
    if not os.path.exists(best_model):
        raise FileNotFoundError(f"Impossible de trouver le modèle : {best_model}")
    
    cmd = [
        sys.executable, "-u", "scripts/evaluate_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--split", "validation",
        "--checkpoint", best_model,
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", output_target
    ]
    
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
        
    process.stdout.close()
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("L'évaluation de validation a échoué.")
    print("\nÉvaluation Validation terminée.")
    
    if GENERATE_REPORT:
        print("\nGénération des rapports visuels...")
        env["MPLBACKEND"] = "Agg"
        rep_cmd = [
            sys.executable, "-u", "scripts/generate_resnet50v2_gtex_report.py",
            "--results-dir", output_target,
            "--class-mapping", f"{DATASET_DIR}/metadata/class_mapping.json"
        ]
        subprocess.run(rep_cmd, cwd=PROJECT_DIR, env=env, check=True)
        print("Rapports générés.")
else:
    print("RUN_VALIDATION_EVALUATION = False. Ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_FINAL_TEST:
    print("Démarrage de l'évaluation sur le split TEST...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    
    output_target = f"{OUTPUT_DIR}/smoke_test" if RUN_SMOKE_TEST else OUTPUT_DIR
    best_model = f"{output_target}/checkpoints/best_model.keras"
    
    cmd = [
        sys.executable, "-u", "scripts/evaluate_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--split", "test",
        "--checkpoint", best_model,
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", output_target
    ]
    
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
        
    process.stdout.close()
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("L'évaluation de TEST a échoué.")
        
    if GENERATE_REPORT:
        env["MPLBACKEND"] = "Agg"
        rep_cmd = [
            sys.executable, "-u", "scripts/generate_resnet50v2_gtex_report.py",
            "--results-dir", output_target,
            "--class-mapping", f"{DATASET_DIR}/metadata/class_mapping.json"
        ]
        subprocess.run(rep_cmd, cwd=PROJECT_DIR, env=env, check=True)
else:
    print("RUN_FINAL_TEST = False. Le set de TEST reste scellé.")


In [ ]:
import json
import datetime
import shutil
import sys

# Generate RUN_MANIFEST.json
manifest = {
    "commit": get_git_commit(),
    "python_version": sys.version,
    "tensorflow_version": tf.__version__ if 'tf' in globals() else "unknown",
    "gpu": gpus[0].name if 'gpus' in globals() and gpus else "None",
    "seed": 42,
    "config": CONFIG_PATH,
    "dataset_dir": str(DATASET_DIR) if 'DATASET_DIR' in globals() else "unknown",
    "date": datetime.datetime.now().isoformat(),
    "status": {
        "train": RUN_TRAINING,
        "validation": RUN_VALIDATION_EVALUATION,
        "test": RUN_FINAL_TEST
    }
}

manifest_path = os.path.join(OUTPUT_DIR, "RUN_MANIFEST.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("Compression des résultats...")
shutil.make_archive(
    "/kaggle/working/resnet50v2-gtex-11-exp-a-results",
    "zip",
    OUTPUT_DIR
)
print("Archive ZIP créée avec succès : /kaggle/working/resnet50v2-gtex-11-exp-a-results.zip")
print("Contenu de /kaggle/working :")
!ls -la /kaggle/working
